# Step 3 — Dual-Process MuJoCo Control (live UnifoLM + bridges)

Run the GIL-free dual-process loop in the notebook (same code as the CLI).

| Setting | Paper run | Smoke test |
|---------|-----------|------------|
| `MOCK` | **`False`** (live UnifoLM) | `True` |
| `BRIDGE` | `esn` then `zoh` then `linear` | `esn` |

- **Process A (VLA):** UnifoLM-VLA-Base @ ~2 Hz → EE→joint IK
- **Process B:** 100 Hz MuJoCo + bridge (`esn` / `zoh` / `linear`)

Reports → `results/step3_dual_thread/dual_thread_report_{bridge}_{live|mock}.json`

Offline ZOH/linear table (no VLA GPU): `step3_control_baselines.ipynb`  
One-shot suite: `step3_sim_comparison.ipynb`


In [ ]:
from pathlib import Path
import os
import sys

# Headless MuJoCo rendering on servers without DISPLAY.
os.environ.setdefault("MUJOCO_GL", "egl")

NOTEBOOK_DIR = Path.cwd().resolve()
if NOTEBOOK_DIR.name == "notebooks":
    RESEARCH_DIR = NOTEBOOK_DIR.parent
elif (NOTEBOOK_DIR / "src" / "step3_dual_thread_mujoco.py").is_file():
    RESEARCH_DIR = NOTEBOOK_DIR
else:
    RESEARCH_DIR = NOTEBOOK_DIR / "research_summer_2026" / "research"
    if not RESEARCH_DIR.is_dir():
        RESEARCH_DIR = NOTEBOOK_DIR / "research"

RESEARCH_DIR = RESEARCH_DIR.resolve()
assert (RESEARCH_DIR / "src").is_dir(), f"src package not found under: {RESEARCH_DIR}"

os.chdir(RESEARCH_DIR)
if str(RESEARCH_DIR) not in sys.path:
    sys.path.insert(0, str(RESEARCH_DIR))

print(f"Research root : {RESEARCH_DIR}")
print(f"Results go to : {RESEARCH_DIR / 'results' / 'step3_dual_thread'}")


In [ ]:
# ── Configuration ───────────────────────────────────────────
# Paper timing: MOCK=False. Smoke test only: MOCK=True.
MOCK = False
BRIDGE = "esn"                 # "esn" | "zoh" | "linear"  — re-run cell for each
DURATION_S = 10.0
CONTROL_HZ = 100.0
VLA_HZ = 2.0
PROFILE = False
PROFILE_STEPS = 200
DEVICE = "cuda"
RECORD_VIDEO = False           # full wipe video → Step 4 notebook
VIDEO_FPS = 60.0

INSTRUCTION = "Wipe the table with the cloth."
UNNORM_KEY = "g1_wipe_table"
INIT_EPISODE = 0
MJCF_PATH = None
ESN_CHECKPOINT = None          # None → models/esn_cuda_ridge best ckpt

print(f"MOCK={MOCK} | BRIDGE={BRIDGE} | duration={DURATION_S}s")
print(f"control={CONTROL_HZ:.0f} Hz | VLA={VLA_HZ:.0f} Hz | record_video={RECORD_VIDEO}")
if MOCK:
    print("WARNING: MOCK=True is a smoke test — set MOCK=False for paper timing.")


In [ ]:
import json
import logging
import multiprocessing as mp
from pathlib import Path

import torch

from src.paths import results_path
from src.step3_dual_thread_mujoco import (
    DualProcessConfig,
    DualProcessController,
    MAX_DURATION_S,
    MAX_STEP_MS_THRESHOLD,
    load_esn_checkpoint_metadata,
    print_run_summary,
    resolve_esn_checkpoint,
    resolve_mjcf_path,
)

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required (target: Tesla V100 / Jetson later).")
if DURATION_S > MAX_DURATION_S:
    raise ValueError(f"DURATION_S={DURATION_S} exceeds MAX_DURATION_S={MAX_DURATION_S}")
if BRIDGE not in ("esn", "zoh", "linear"):
    raise ValueError(f"BRIDGE must be esn|zoh|linear, got {BRIDGE!r}")

print(f"Device: cuda ({torch.cuda.get_device_name(torch.device(DEVICE))})")

mjcf = resolve_mjcf_path(MJCF_PATH)
out_dir = results_path("step3_dual_thread")
vla_tag = "mock" if MOCK else "live"
video_path = out_dir / f"table_wipe_{BRIDGE}_{vla_tag}.mp4"

esn_meta = {}
ckpt = Path("unused")
if BRIDGE == "esn":
    ckpt = resolve_esn_checkpoint(ESN_CHECKPOINT)
    esn_meta = load_esn_checkpoint_metadata(ckpt)
    print(f"ESN checkpoint (Step 2): {ckpt}")
    if esn_meta.get("metrics"):
        m = esn_meta["metrics"]
        print(
            f"  Step 2 metrics: MSE={m['mse']:.2e} jerk={m['jerk']:.2e} "
            f"α={m['leaky_rate']:.2f} λ={m['ridge_alpha']:.1e} "
            f"dataset={esn_meta.get('dataset_id', '?')}"
        )
else:
    print(f"Bridge={BRIDGE} — ESN checkpoint not required.")

print(f"MJCF: {mjcf}")
print(f"Report will be: dual_thread_report_{BRIDGE}_{vla_tag}.json")


In [ ]:
mp.set_start_method("spawn", force=True)

config = DualProcessConfig(
    mjcf_path=mjcf,
    esn_checkpoint=str(ckpt),
    mock=MOCK,
    duration_s=DURATION_S,
    control_hz=CONTROL_HZ,
    vla_hz=VLA_HZ,
    instruction=INSTRUCTION,
    device=DEVICE,
    profile=PROFILE,
    profile_steps=PROFILE_STEPS,
    record_video=RECORD_VIDEO,
    video_path=video_path if RECORD_VIDEO else None,
    video_fps=VIDEO_FPS,
    unnorm_key=UNNORM_KEY,
    init_episode=INIT_EPISODE,
    use_wipe_table_scene=True,
    bridge=BRIDGE,
)

print(f"Starting dual-process | bridge={BRIDGE} | mock={MOCK} | {DURATION_S:.1f}s ...")
controller = DualProcessController(config)
stats = controller.run()

report = {
    "architecture": "multiprocessing",
    "task": "g1_wipe_table",
    "bridge": BRIDGE,
    "mock_vla": MOCK,
    "mjcf": str(mjcf),
    "esn_checkpoint": str(ckpt) if BRIDGE == "esn" else None,
    "duration_s": DURATION_S,
    "control_hz_target": CONTROL_HZ,
    "vla_hz_target": VLA_HZ,
    "steps": stats.steps,
    "mean_step_ms": stats.mean_step_ms,
    "max_step_ms": stats.max_step_ms,
    "max_step_ms_steady": stats.max_step_ms_steady,
    "p99_step_ms": stats.p99_step_ms,
    "achieved_control_hz": stats.esn_hz,
    "achieved_esn_hz": stats.esn_hz if BRIDGE == "esn" else None,
    "vla_ticks": stats.vla_ticks,
    "vla_register_sequence": stats.vla_seq_final,
    "gil_bypass_ok": stats.gil_bypass_ok,
    "max_step_ms_threshold": MAX_STEP_MS_THRESHOLD,
    "video_path": stats.video_path,
}

report_path = out_dir / f"dual_thread_report_{BRIDGE}_{vla_tag}.json"
out_dir.mkdir(parents=True, exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)
if BRIDGE == "esn":
    legacy = out_dir / "dual_thread_report.json"
    with open(legacy, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

print_run_summary(
    stats,
    control_hz=CONTROL_HZ,
    vla_hz=VLA_HZ,
    report_path=report_path,
    profile=PROFILE,
    bridge=BRIDGE,
    mock=MOCK,
)
print(f"Saved: {report_path}")
print("Next: set BRIDGE='zoh' (then 'linear'), re-run config + this cell.")


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

summary = pd.DataFrame([report]).T
summary.columns = ["value"]
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bars = axes[0].bar(
    ["Target Hz", "Achieved Hz", "Target VLA Hz", "VLA ticks"],
    [CONTROL_HZ, stats.esn_hz, VLA_HZ, stats.vla_ticks],
    color=["#9E9E9E", "#4CAF50", "#9E9E9E", "#2196F3"],
)
axes[0].axhline(CONTROL_HZ, color="#F44336", ls="--", lw=1.5, label=f"{CONTROL_HZ:.0f} Hz target")
axes[0].set_ylabel("Hz / count")
axes[0].set_title(f"Throughput ({BRIDGE}, {'mock' if MOCK else 'live'})")
axes[0].legend()
for b in bars:
    axes[0].text(
        b.get_x() + b.get_width() / 2,
        b.get_height(),
        f"{b.get_height():.1f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

lat_colors = [
    "#4CAF50" if stats.max_step_ms_steady < MAX_STEP_MS_THRESHOLD else "#F44336",
    "#2196F3",
    "#9C27B0",
]
axes[1].bar(
    ["Steady max (ms)", "Mean step (ms)", "P99 (ms)"],
    [stats.max_step_ms_steady, stats.mean_step_ms, stats.p99_step_ms],
    color=lat_colors,
)
axes[1].axhline(MAX_STEP_MS_THRESHOLD, color="#F44336", ls="--", lw=1.5, label=f"{MAX_STEP_MS_THRESHOLD:.0f} ms budget")
axes[1].set_ylabel("Latency (ms)")
axes[1].set_title("Control-loop latency")
axes[1].legend()

plt.tight_layout()
fig_path = out_dir / f"dual_thread_summary_{BRIDGE}_{vla_tag}.png"
fig.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print(f"Figure saved: {fig_path}")

if stats.video_path and Path(stats.video_path).is_file():
    from IPython.display import HTML, Video
    print(f"Benchmark video: {stats.video_path}")
    try:
        display(Video(stats.video_path, embed=True, width=640, html_attributes="controls loop"))
    except Exception:
        display(HTML(
            f'<video width="640" controls loop>'
            f'<source src="{stats.video_path}" type="video/mp4"></video>'
        ))
